In [157]:
import sys

print(sys.executable)

C:\Users\RACHNA\AppData\Local\Programs\Python\Python311\python.exe


In [158]:
import sys

!{sys.executable} -m pip install sentence-transformers
!{sys.executable} -m pip install faiss-cpu scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\RACHNA\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\RACHNA\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [159]:
import sys

!{sys.executable} -m pip install tf-keras


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\RACHNA\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [160]:
from sentence_transformers import SentenceTransformer

In [161]:
!pip install sentence-transformers
!pip install faiss-cpu
!pip install scikit-learn
!pip install pandas numpy matplotlib

In [162]:
!pip install sentence-transformers faiss-cpu scikit-learn

In [163]:
import re

import numpy as np
import pandas as pd

In [164]:
df = pd.read_csv("data/dataset.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (12041, 12)

Columns:
['category_1', 'category_2', 'category_3', 'title', 'product_rating', 'selling_price', 'mrp', 'seller_name', 'seller_rating', 'description', 'highlights', 'image_links']


,category_1,category_2,category_3,title,product_rating,selling_price,mrp,seller_name,seller_rating,description,highlights,image_links
0,"Sports, Books and More",Sports,Cricket,ITWOSERVICES CRICKET NET 100X10 CRICKET NET NY...,4.4,"₹1,615","₹4,000",I2SERVICES,4.4,NaN,Cricket Practice Net NYLON HDPE Material W x H...,https://rukminim1.flixcart.com/image/612/612/x...
1,"Sports, Books and More",Sports,Cricket,ITWOSERVICES CRICKET NET GROUND BOUNDARY NET 1...,4.4,₹152,₹600,I2SERVICES,4.4,10 X 10 GREEN CRICKET NET HDPE NYLON.,Cricket HDPE NYLON Material W x H x D: 3.048 x...,https://rukminim1.flixcart.com/image/612/612/x...
2,"Sports, Books and More",Sports,Cricket,VICTORY Medium Weight ( Pack of 1 ) Rubber Cri...,3.7,₹59,₹199,VictoryOutlets,4.7,NaN,Cricket Rubber Ball Weight: 110 g,https://rukminim1.flixcart.com/image/612/612/x...
3,"Sports, Books and More",Sports,Cricket,VICTORY Cricket Wind Ball (Pack of 1) - Made i...,3.8,₹75,₹299,VictoryOutlets,4.7,NaN,Cricket Synthetic Ball Weight: 110 g,https://rukminim1.flixcart.com/image/612/612/k...
4,"Sports, Books and More",Sports,Cricket,CEAT Hitman Full Size Double Blade Poplar Cric...,3.4,₹329,"₹1,399",IndiaFit,4.7,The Ceat Poplar Willow Cricket Bat has been de...,Age Group 15+ Yrs Blade Made of Poplar Willow ...,https://rukminim1.flixcart.com/image/612/612/j...


In [165]:
df = df.fillna("")


def create_document(row):
    return f"""
    Product: {row["title"]}

    Category:
    {row["category_1"]} > {row["category_2"]} > {row["category_3"]}

    Description:
    {row["description"]}

    Highlights:
    {row["highlights"]}

    Selling Price:
    {row["selling_price"]}

    Product Rating:
    {row["product_rating"]}

    Seller Rating:
    {row["seller_rating"]}
    """


documents = df.apply(create_document, axis=1).tolist()

print("Total documents:", len(documents))
print("\nSample document:\n")
print(documents[0][:1000])

Total documents: 12041

Sample document:


    Product: ITWOSERVICES CRICKET NET 100X10 CRICKET NET NYLON HDPE Cricket Net  (Green)

    Category:
    Sports, Books and More > Sports > Cricket 

    Description:
    

    Highlights:
    Cricket Practice Net NYLON HDPE Material W x H x D: 10 x 10

    Selling Price:
    ₹1,615

    Product Rating:
    4.4

    Seller Rating:
    4.4
    


In [166]:
print("Category 1:", df["category_1"].nunique())
print("Category 2:", df["category_2"].nunique())
print("Category 3:", df["category_3"].nunique())

print("\nTop Category 1 Values:\n")
print(df["category_1"].value_counts().head(15))

Category 1: 6
Category 2: 77
Category 3: 300

Top Category 1 Values:

category_1
Women's wear              2422
Men's wear                2360
Bady and Kids             2338
Home and Furniture        2120
Sports, Books and More    1820
Electronics                981
Name: count, dtype: int64


In [167]:
def clean_text(text):
    text = text.lower()

    text = re.sub(r"\S+@\S+", "", text)

    text = re.sub(r"http\S+|www\S+", "", text)

    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [168]:
clean_documents = [clean_text(doc) for doc in documents]

print(clean_documents[0][:500])

product itwoservices cricket net 100x10 cricket net nylon hdpe cricket net green category sports books and more sports cricket description highlights cricket practice net nylon hdpe material w x h x d 10 x 10 selling price 1 615 product rating 4 4 seller rating 4 4


In [169]:
with open("clean_documents.txt", "w", encoding="utf-8") as f:
    for doc in clean_documents:
        f.write(doc.replace("\n", " ") + "\n")

In [170]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [171]:
embeddings = model.encode(clean_documents, show_progress_bar=True)

Batches:   0%|          | 0/377 [00:00<?, ?it/s]

In [172]:
print(embeddings.shape)

(12041, 384)


In [173]:
import faiss

In [174]:
embeddings = embeddings.astype("float32")

In [175]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

In [176]:
index.add(embeddings)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 12041


In [177]:
query = "wireless bluetooth earbuds"

query_embedding = model.encode([query])
query_embedding = query_embedding.astype("float32")

D, I = index.search(query_embedding, k=20)

results = []

for rank, idx in enumerate(I[0]):
    distance = D[0][rank]

    rating = float(df.iloc[idx]["product_rating"])

    similarity = 1 / (1 + distance)

    final_score = 0.8 * similarity + 0.2 * (rating / 5.0)

    results.append((idx, distance, final_score))

# Sort by final score
results.sort(key=lambda x: x[2], reverse=True)

for rank, (idx, distance, final_score) in enumerate(results):
    print("=" * 60)

    print("Rank:", rank + 1)

    print("Title:", df.iloc[idx]["title"])

    print("Category:", df.iloc[idx]["category_1"])

    print("Price:", df.iloc[idx]["selling_price"])

    print("Rating:", df.iloc[idx]["product_rating"])

    print("Distance:", distance)

    print("Final Score:", round(final_score, 4))

Rank: 1
Title: Winsumm combo offer K1-Pack 2 Wireless Bluetooth In Ear Headset with Mic (Black) Smart Headphones  (Wireless)
Category: Electronics
Price: ₹299
Rating: 5.0
Distance: 0.86400676
Final Score: 0.6292
Rank: 2
Title: HAROON i12 Pro 100% Original Bluetooth Earbuds (Yellow) Smart Headphones  (Wireless)
Category: Electronics
Price: ₹589
Rating: 4.2
Distance: 0.7348221
Final Score: 0.6291
Rank: 3
Title: HAROON i12 Pro 100% Original Bluetooth Earbuds (Pink) Smart Headphones  (Wireless)
Category: Electronics
Price: ₹589
Rating: 4.2
Distance: 0.7350958
Final Score: 0.6291
Rank: 4
Title: wishmechstore T2 PRO Bluetooth Earbuds in-Ear True Wireless Earbuds with Power Bank function Smart Headphones  (Wireless)
Category: Electronics
Price: ₹899
Rating: 4.5
Distance: 0.8054637
Final Score: 0.6231
Rank: 5
Title: HAROON 100% Original TWS-L21 Bluetooth Earphone Earbuds Smart Headphones  (Wireless)
Category: Electronics
Price: ₹649
Rating: 3.6
Distance: 0.6944001
Final Score: 0.6161
Rank: 6
T

In [178]:
product_data = df[
    ["title", "category_1", "selling_price", "product_rating", "seller_rating"]
]

In [179]:
np.save("product_embeddings.npy", embeddings)

In [180]:
faiss.write_index(index, "product_index.faiss")

In [181]:
!pip install rank-bm25

In [182]:
import sys

print(sys.executable)

C:\Users\RACHNA\AppData\Local\Programs\Python\Python311\python.exe


In [183]:
!{sys.executable} -m pip install rank-bm25


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\RACHNA\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [184]:
from rank_bm25 import BM25Okapi

In [185]:
tokenized_docs = [doc.split() for doc in clean_documents]

In [186]:
bm25 = BM25Okapi(tokenized_docs)

In [187]:
query = "wireless bluetooth earbuds"

tokenized_query = query.split()

scores = bm25.get_scores(tokenized_query)

In [188]:
top_indices = np.argsort(scores)[::-1][:5]

In [189]:
for idx in top_indices:
    print(product_data.iloc[idx]["title"])

wishmechstore T2 PRO Bluetooth Earbuds in-Ear True Wireless Earbuds with Power Bank function Smart Headphones  (Wireless)
HOLA SynerGe WIRELESS Smart Headphones  (Wireless)
Linsden Bluetooth Earbuds -1 Smart Headphones  (Wireless)
HAROON i12 Pro 100% Original Bluetooth Earbuds (Yellow) Smart Headphones  (Wireless)
HAROON i12 Pro 100% Original Bluetooth Earbuds (Pink) Smart Headphones  (Wireless)


In [190]:
query = "wireless bluetooth earbuds"

query_embedding = model.encode([query])
query_embedding = query_embedding.astype("float32")

D, I = index.search(query_embedding, k=50)

faiss_top_indices = I[0]

print("FAISS RESULTS\n")

for idx in faiss_top_indices:
    print(product_data.iloc[idx]["title"])

FAISS RESULTS

HAROON 100% Original i12 tws Bluetooth Earphone Earbuds Smart Headphones  (Wireless)
HAROON 100% Original TWS-L21 Bluetooth Earphone Earbuds Smart Headphones  (Wireless)
Linsden Bluetooth Earbuds -1 Smart Headphones  (Wireless)
HAROON i12 Pro 100% Original Bluetooth Earbuds (Yellow) Smart Headphones  (Wireless)
HAROON i12 Pro 100% Original Bluetooth Earbuds (Pink) Smart Headphones  (Wireless)
Zrose Bluetooth Wireless in Ear Earphones with Mic Smart Headphones  (Wireless)
wishmechstore T2 PRO Bluetooth Earbuds in-Ear True Wireless Earbuds with Power Bank function Smart Headphones  (Wireless)
HOLA SynerGe WIRELESS Smart Headphones  (Wireless)
AF HBS-730 Neckband Bluetooth Headphones Wireless Sport Stereo Headsets Handsfree with Microphone for Android, (Black) Smart Headphones  (Wireless)
BRD 10.0 A HEADPHONE Smart Headphones  (Wireless)
Winsumm combo offer K1-Pack 2 Wireless Bluetooth In Ear Headset with Mic (Black) Smart Headphones  (Wireless)
NIRUM Wireless Bluetooth 4.1

In [191]:
tokenized_query = query.split()

scores = bm25.get_scores(tokenized_query)

bm25_top_indices = np.argsort(scores)[::-1][:50]

print("BM25 RESULTS\n")

for idx in bm25_top_indices:
    print(product_data.iloc[idx]["title"])

BM25 RESULTS

wishmechstore T2 PRO Bluetooth Earbuds in-Ear True Wireless Earbuds with Power Bank function Smart Headphones  (Wireless)
HOLA SynerGe WIRELESS Smart Headphones  (Wireless)
Linsden Bluetooth Earbuds -1 Smart Headphones  (Wireless)
HAROON i12 Pro 100% Original Bluetooth Earbuds (Yellow) Smart Headphones  (Wireless)
HAROON i12 Pro 100% Original Bluetooth Earbuds (Pink) Smart Headphones  (Wireless)
HAROON 100% Original i12 tws Bluetooth Earphone Earbuds Smart Headphones  (Wireless)
HAROON 100% Original TWS-L21 Bluetooth Earphone Earbuds Smart Headphones  (Wireless)
Kundra uper Quality Stereo Wireless Electronics Bluetooth Outdoor Activities Bluetooth Wireless Smart Sunglasses With Hands-Free Calling Function Wireless Sports Sunglasses Bluetooth Earphone Deep Bass Lightweight Bluetooth Headset Sunglasses Headphone Wireless Bluetooth Headphones With Good Touch Function (Smart Glasses, Black)  (Smart Glasses, black)
Kundra uper Quality Stereo Wireless Electronics Bluetooth Outd

In [192]:
from collections import defaultdict

In [193]:
rrf_scores = defaultdict(float)

k = 60

for rank, idx in enumerate(bm25_top_indices):
    rrf_scores[idx] += 1 / (k + rank + 1)

for rank, idx in enumerate(faiss_top_indices):
    rrf_scores[idx] += 1 / (k + rank + 1)

In [194]:
final_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

In [195]:
print("RRF RESULTS\n")

for idx, score in final_results[:50]:
    print("=" * 60)

    print("Title:", product_data.iloc[idx]["title"])

    print("Score:", round(score, 4))

RRF RESULTS

Title: Linsden Bluetooth Earbuds -1 Smart Headphones  (Wireless)
Score: 0.0317
Title: HAROON 100% Original i12 tws Bluetooth Earphone Earbuds Smart Headphones  (Wireless)
Score: 0.0315
Title: wishmechstore T2 PRO Bluetooth Earbuds in-Ear True Wireless Earbuds with Power Bank function Smart Headphones  (Wireless)
Score: 0.0313
Title: HAROON i12 Pro 100% Original Bluetooth Earbuds (Yellow) Smart Headphones  (Wireless)
Score: 0.0312
Title: HAROON 100% Original TWS-L21 Bluetooth Earphone Earbuds Smart Headphones  (Wireless)
Score: 0.0311
Title: HOLA SynerGe WIRELESS Smart Headphones  (Wireless)
Score: 0.0308
Title: HAROON i12 Pro 100% Original Bluetooth Earbuds (Pink) Smart Headphones  (Wireless)
Score: 0.0308
Title: Zrose Bluetooth Wireless in Ear Earphones with Mic Smart Headphones  (Wireless)
Score: 0.029
Title: bounteous Top Quality On The Ear Bluetooth Headphone with Foldable Adjustable Headphone Mp3 Headphones Best Bluetooth Audio Player Stereo Wireless Electronics Bluet

In [196]:
print(product_data.columns)

Index(['title', 'category_1', 'selling_price', 'product_rating',
       'seller_rating'],
      dtype='object')


In [197]:
print(product_data["category_1"].unique())

['Sports, Books and More' 'Electronics' "Men's wear" "Women's wear"
 'Bady and Kids' 'Home and Furniture']


In [198]:
query = "wireless bluetooth earbuds"

max_rrf = max(score for _, score in final_results)

reranked_results = []

for idx, rrf_score in final_results:
    normalized_rrf = rrf_score / max_rrf

    rating = float(product_data.iloc[idx]["product_rating"])

    seller_rating = float(product_data.iloc[idx]["seller_rating"])

    rating_score = rating / 5.0
    seller_score = seller_rating / 5.0

    final_score = 0.60 * normalized_rrf + 0.30 * rating_score + 0.10 * seller_score

    # Category boost
    category1 = str(product_data.iloc[idx]["category_1"]).lower()

    if "earbuds" in query.lower() and category1 == "electronics":
        final_score *= 1.10

    reranked_results.append((idx, final_score, rating, seller_rating))

# Sort by final score
reranked_results.sort(key=lambda x: x[1], reverse=True)

In [199]:
reranked_results.sort(key=lambda x: x[1], reverse=True)

In [200]:
for idx, score, rating, seller_rating in reranked_results[:10]:
    print("=" * 60)

    print("Title:", product_data.iloc[idx]["title"])

    print("Rating:", rating)

    print("Final Score:", round(score, 4))

Title: wishmechstore T2 PRO Bluetooth Earbuds in-Ear True Wireless Earbuds with Power Bank function Smart Headphones  (Wireless)
Rating: 4.5
Final Score: 1.0559
Title: HAROON i12 Pro 100% Original Bluetooth Earbuds (Yellow) Smart Headphones  (Wireless)
Rating: 4.2
Final Score: 1.0193
Title: HAROON i12 Pro 100% Original Bluetooth Earbuds (Pink) Smart Headphones  (Wireless)
Rating: 4.2
Final Score: 1.0093
Title: HOLA SynerGe WIRELESS Smart Headphones  (Wireless)
Rating: 3.6
Final Score: 0.9799
Title: HAROON 100% Original TWS-L21 Bluetooth Earphone Earbuds Smart Headphones  (Wireless)
Rating: 3.6
Final Score: 0.9756
Title: Linsden Bluetooth Earbuds -1 Smart Headphones  (Wireless)
Rating: 3.4
Final Score: 0.9592
Title: HAROON 100% Original i12 tws Bluetooth Earphone Earbuds Smart Headphones  (Wireless)
Rating: 3.1
Final Score: 0.9528
Title: Zrose Bluetooth Wireless in Ear Earphones with Mic Smart Headphones  (Wireless)
Rating: 3.5
Final Score: 0.9272
Title: Kundra uper Quality Stereo Wirel

In [201]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [202]:
top_candidates = reranked_results[:20]

pairs = []

for idx, score, rating, seller_rating in top_candidates:
    product_text = clean_documents[idx]

    pairs.append((query, product_text))

In [203]:
ce_scores = cross_encoder.predict(pairs)

In [204]:
final_cross_results = []

for i, (idx, score, rating, seller_rating) in enumerate(top_candidates):
    ce_score = ce_scores[i]

    final_cross_results.append((idx, ce_score))

In [205]:
final_cross_results.sort(key=lambda x: x[1], reverse=True)

In [206]:
for idx, ce_score in final_cross_results[:10]:
    print("=" * 60)

    print(product_data.iloc[idx]["title"])

    print("Cross Score:", round(float(ce_score), 4))

wishmechstore T2 PRO Bluetooth Earbuds in-Ear True Wireless Earbuds with Power Bank function Smart Headphones  (Wireless)
Cross Score: 7.512
HAROON 100% Original i12 tws Bluetooth Earphone Earbuds Smart Headphones  (Wireless)
Cross Score: 6.8564
HAROON 100% Original TWS-L21 Bluetooth Earphone Earbuds Smart Headphones  (Wireless)
Cross Score: 6.8327
HAROON i12 Pro 100% Original Bluetooth Earbuds (Pink) Smart Headphones  (Wireless)
Cross Score: 6.8311
HAROON i12 Pro 100% Original Bluetooth Earbuds (Yellow) Smart Headphones  (Wireless)
Cross Score: 6.8004
Linsden Bluetooth Earbuds -1 Smart Headphones  (Wireless)
Cross Score: 6.7626
HOLA SynerGe WIRELESS Smart Headphones  (Wireless)
Cross Score: 5.0259
Zrose Bluetooth Wireless in Ear Earphones with Mic Smart Headphones  (Wireless)
Cross Score: 4.4909
Winsumm combo offer K1-Pack 2 Wireless Bluetooth In Ear Headset with Mic (Black) Smart Headphones  (Wireless)
Cross Score: 4.1411
Linsden K1 bluetooth Smart Headphones  (Wireless)
Cross Score:

In [208]:
product_data[["selling_price", "price_num"]].head()

,selling_price,price_num
0,"₹1,615",1615.0
1,₹152,152.0
2,₹59,59.0
3,₹75,75.0
4,₹329,329.0


In [209]:
query = "best wireless earbuds under 1000 rating above 4"

print(parse_query(query))

{'max_price': 1000, 'min_rating': 4.0, 'product_type': 'earbuds'}


In [210]:
def parse_query(query):

    result = {}

    # Price
    price_match = re.search(r"under\s+(\d+)", query.lower())

    if price_match:
        result["max_price"] = int(price_match.group(1))

    # Rating
    rating_match = re.search(r"rating\s+above\s+(\d+)", query.lower())

    if rating_match:
        result["min_rating"] = float(rating_match.group(1))

    # Product Type
    product_types = [
        "earbuds",
        "headphones",
        "speaker",
        "soundbar",
        "watch",
        "mobile",
        "laptop",
    ]

    for product in product_types:
        if product in query.lower():
            result["product_type"] = product
            break

    return result

In [214]:
product_data = df

In [215]:
product_data["price_num"] = (
    product_data["selling_price"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
)

product_data["price_num"] = pd.to_numeric(product_data["price_num"], errors="coerce")

In [216]:
constraints = parse_query(query)

print(constraints)

{'max_price': 1000, 'min_rating': 4.0, 'product_type': 'earbuds'}


In [217]:
constraints = parse_query(query)

filtered_results = []

for idx, score, rating, seller_rating in reranked_results:
    price = product_data.iloc[idx]["price_num"]

    title = str(product_data.iloc[idx]["title"]).lower()

    if "max_price" in constraints:
        if price > constraints["max_price"]:
            continue

    if "min_rating" in constraints:
        if rating < constraints["min_rating"]:
            continue

    if "product_type" in constraints:
        if constraints["product_type"] not in title:
            continue

    filtered_results.append((idx, score, rating, seller_rating))

In [218]:
semantic_cache = {}

In [219]:
def search(query):

    constraints = parse_query(query)

    filtered_results = []

    for idx, score, rating, seller_rating in reranked_results:
        price = product_data.iloc[idx]["price_num"]

        title = str(product_data.iloc[idx]["title"]).lower()

        if "max_price" in constraints:
            if price > constraints["max_price"]:
                continue

        if "min_rating" in constraints:
            if rating < constraints["min_rating"]:
                continue

        if "product_type" in constraints:
            if constraints["product_type"] not in title:
                continue

        filtered_results.append((idx, score, rating, seller_rating))

    # Remove duplicates
    seen = set()
    unique_results = []

    for idx, score, rating, seller_rating in filtered_results:
        title = product_data.iloc[idx]["title"]

        if title not in seen:
            seen.add(title)

            unique_results.append((idx, score, rating, seller_rating))

    return unique_results

In [229]:
for idx, score, rating, seller_rating in results[:10]:
    print("=" * 60)

    print(product_data.iloc[idx]["title"])

    print("Price:", product_data.iloc[idx]["selling_price"])

    print("Rating:", product_data.iloc[idx]["product_rating"])

    print("Seller Rating:", product_data.iloc[idx]["seller_rating"])

    print("Score:", round(score, 4))

wishmechstore T2 PRO Bluetooth Earbuds in-Ear True Wireless Earbuds with Power Bank function Smart Headphones  (Wireless)
Price: ₹899
Rating: 4.5
Seller Rating: 4.9
Score: 1.0559
HAROON i12 Pro 100% Original Bluetooth Earbuds (Yellow) Smart Headphones  (Wireless)
Price: ₹589
Rating: 4.2
Seller Rating: 4.2
Score: 1.0193
HAROON i12 Pro 100% Original Bluetooth Earbuds (Pink) Smart Headphones  (Wireless)
Price: ₹589
Rating: 4.2
Seller Rating: 4.2
Score: 1.0093
HAROON 100% Original TWS-L21 Bluetooth Earphone Earbuds Smart Headphones  (Wireless)
Price: ₹649
Rating: 3.6
Seller Rating: 4.2
Score: 0.9756
Linsden Bluetooth Earbuds -1 Smart Headphones  (Wireless)
Price: ₹346
Rating: 3.4
Seller Rating: 3.4
Score: 0.9592
HAROON 100% Original i12 tws Bluetooth Earphone Earbuds Smart Headphones  (Wireless)
Price: ₹499
Rating: 3.1
Seller Rating: 4.2
Score: 0.9528


In [226]:
print(parse_query("wireless earbuds under 1000"))

{'max_price': 1000, 'product_type': 'earbuds'}


In [230]:
results = search("wireless earbuds under 1000")

print(len(results))

for r in results[:10]:
    print(r)

6
(np.int64(2019), 1.0559181795938342, 4.5, 4.9)
(np.int64(2014), 1.0192875000000001, 4.2, 4.2)
(np.int64(1994), 1.009292307692308, 4.2, 4.2)
(np.int64(2017), 0.975621088107848, 3.6, 4.2)
(np.int64(2022), 0.9591999999999999, 3.4, 3.4)
(np.int64(1995), 0.9528196721311478, 3.1, 4.2)


In [231]:
retrieve("wireless earbuds under 1000")

Searching for: wireless earbuds under 1000


[(np.int64(2019), 1.0559181795938342, 4.5, 4.9),
 (np.int64(2014), 1.0192875000000001, 4.2, 4.2),
 (np.int64(1994), 1.009292307692308, 4.2, 4.2)]

In [232]:
print(parse_query("wireless earbuds under 500"))

{'max_price': 500, 'product_type': 'earbuds'}


In [233]:
results = search("wireless earbuds under 500")

print(len(results))

2


In [250]:
results = search("wireless earbuds under 1000")

print(results)

[(np.int64(2019), 1.0559181795938342, 4.5, 4.9), (np.int64(2014), 1.0192875000000001, 4.2, 4.2), (np.int64(1994), 1.009292307692308, 4.2, 4.2), (np.int64(2017), 0.975621088107848, 3.6, 4.2), (np.int64(2022), 0.9591999999999999, 3.4, 3.4), (np.int64(1995), 0.9528196721311478, 3.1, 4.2)]


In [251]:
print(type(results))
print(results[0])
print(len(results))

<class 'list'>
(np.int64(2019), 1.0559181795938342, 4.5, 4.9)
6


In [253]:
print(semantic_cache.keys())

dict_keys(['wireless earbuds under 1000'])


In [254]:
query = "wireless earbuds under 1000"

if query in semantic_cache:
    print("CACHE HIT")
    results = semantic_cache[query]
else:
    print("CACHE MISS")
    results = filtered_results
    semantic_cache[query] = results

CACHE HIT


In [255]:
results = search("best wireless earbuds under 1000 rating above 4")

In [256]:
from fastapi import FastAPI

app = FastAPI()


@app.post("/search")
def search_api(query: str):
    return search(query)

In [258]:
results = search("best wireless earbuds under 1000 rating above 4")

print(len(results))

3


In [259]:
results = search("wireless earbuds under 500")

for idx, score, rating, seller_rating in results:
    print("=" * 60)

    print(product_data.iloc[idx]["title"])

    print("Price:", product_data.iloc[idx]["selling_price"])

Linsden Bluetooth Earbuds -1 Smart Headphones  (Wireless)
Price: ₹346
HAROON 100% Original i12 tws Bluetooth Earphone Earbuds Smart Headphones  (Wireless)
Price: ₹499


In [260]:
test_queries = [
    "wireless earbuds under 1000",
    "bluetooth speaker",
    "wireless headphones",
    "soundbar under 1000",
]

In [261]:
import inspect

print(inspect.signature(retrieve_dense))
print(inspect.getsource(retrieve_dense))

(query)
def retrieve_dense(query):
    pass



In [262]:
print(inspect.getsource(retrieve_sparse))

def retrieve_sparse(query):
    pass



In [263]:
results = search("wireless earbuds under 1000")

for idx, score, rating, seller_rating in results:
    print(product_data.iloc[idx]["title"])

wishmechstore T2 PRO Bluetooth Earbuds in-Ear True Wireless Earbuds with Power Bank function Smart Headphones  (Wireless)
HAROON i12 Pro 100% Original Bluetooth Earbuds (Yellow) Smart Headphones  (Wireless)
HAROON i12 Pro 100% Original Bluetooth Earbuds (Pink) Smart Headphones  (Wireless)
HAROON 100% Original TWS-L21 Bluetooth Earphone Earbuds Smart Headphones  (Wireless)
Linsden Bluetooth Earbuds -1 Smart Headphones  (Wireless)
HAROON 100% Original i12 tws Bluetooth Earphone Earbuds Smart Headphones  (Wireless)


In [264]:
print(f"Found {len(results)} matching products")

Found 6 matching products


In [265]:
print("Seller Rating:", seller_rating)

Seller Rating: 4.2


In [266]:
test_queries = [
    "wireless earbuds under 1000",
    "wireless earbuds under 500",
    "bluetooth speaker",
    "soundbar under 1000",
]

for q in test_queries:
    print("\n" + "=" * 70)
    print("QUERY:", q)
    print("=" * 70)

    results = search(q)

    for idx, score, rating, seller_rating in results[:5]:
        print(product_data.iloc[idx]["title"])
        print("Price:", product_data.iloc[idx]["selling_price"])
        print("Rating:", rating)
        print("Score:", round(score, 4))
        print()


QUERY: wireless earbuds under 1000
wishmechstore T2 PRO Bluetooth Earbuds in-Ear True Wireless Earbuds with Power Bank function Smart Headphones  (Wireless)
Price: ₹899
Rating: 4.5
Score: 1.0559

HAROON i12 Pro 100% Original Bluetooth Earbuds (Yellow) Smart Headphones  (Wireless)
Price: ₹589
Rating: 4.2
Score: 1.0193

HAROON i12 Pro 100% Original Bluetooth Earbuds (Pink) Smart Headphones  (Wireless)
Price: ₹589
Rating: 4.2
Score: 1.0093

HAROON 100% Original TWS-L21 Bluetooth Earphone Earbuds Smart Headphones  (Wireless)
Price: ₹649
Rating: 3.6
Score: 0.9756

Linsden Bluetooth Earbuds -1 Smart Headphones  (Wireless)
Price: ₹346
Rating: 3.4
Score: 0.9592


QUERY: wireless earbuds under 500
Linsden Bluetooth Earbuds -1 Smart Headphones  (Wireless)
Price: ₹346
Rating: 3.4
Score: 0.9592

HAROON 100% Original i12 tws Bluetooth Earphone Earbuds Smart Headphones  (Wireless)
Price: ₹499
Rating: 3.1
Score: 0.9528


QUERY: bluetooth speaker
Runixx Wireless Bluetooth Speaker with Mobile Stand,Dj

In [267]:
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

In [268]:
pca = PCA(n_components=50, random_state=42)

reduced_embeddings = pca.fit_transform(embeddings)

print(reduced_embeddings.shape)

(12041, 50)


In [271]:
import time

bic_scores = []

for k in range(5, 16):
    start = time.time()

    gmm = GaussianMixture(n_components=k, covariance_type="diag", random_state=42)

    gmm.fit(reduced_embeddings)

    bic_scores.append(gmm.bic(reduced_embeddings))

    print(f"k={k} done in {time.time() - start:.1f}s")

k=5 done in 0.8s
k=6 done in 0.9s
k=7 done in 1.2s
k=8 done in 0.9s
k=9 done in 0.9s
k=10 done in 0.9s
k=11 done in 0.9s
k=12 done in 0.9s
k=13 done in 0.7s
k=14 done in 1.3s
k=15 done in 1.4s


In [272]:
best_k = cluster_range[np.argmin(bic_scores)]

print("Best number of clusters:", best_k)

Best number of clusters: 15


In [273]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")

tfidf_matrix = vectorizer.fit_transform(clean_documents)

feature_names = vectorizer.get_feature_names_out()

In [274]:
cluster_probabilities = gmm.predict_proba(reduced_embeddings)

print(cluster_probabilities.shape)

(12041, 15)


In [276]:
cluster_labels = cluster_probabilities.argmax(axis=1)

In [277]:
import numpy as np

for cluster in range(best_k):
    print("\nCluster", cluster)

    cluster_docs = np.where(cluster_labels == cluster)[0]

    cluster_tfidf = tfidf_matrix[cluster_docs].mean(axis=0)

    top_words = np.argsort(cluster_tfidf).tolist()[0][-10:]

    for word in reversed(top_words):
        print(feature_names[word])


Cluster 0
men
wear
rating
product
solid
loungewear
category
description
selling
seller

Cluster 1
steel
ml
kitchen
lunch
stainless
containers
bottle
capacity
box
home

Cluster 2
product
sports
rating
books
fitness
home
school
bag
water
price

Cluster 3
toys
kids
toy
age
years
bady
cm
plastic
game
operated

Cluster 4
mobile
bluetooth
smart
wireless
electronics
usb
product
cable
gb
rating

Cluster 5
women
product
rating
wear
accessories
lingerie
sunglasses
jewellery
sleepware
category

Cluster 6
food
skin
hair
product
men
fragrance
rating
chocolate
nutrition
health

Cluster 7
shoes
foot
wear
men
rating
product
women
footwear
boots
running

Cluster 8
baby
care
kids
bady
feeding
product
wipes
rating
soft
skin

Cluster 9
watches
watch
analog
men
sonata
fastrack
titan
fossil
product
rating

Cluster 10
hair
settings
trimmer
appliances
personal
men
length
care
min
blade

Cluster 11
women
wear
ethnic
product
rating
western
maternity
trousers
bottoms
seller

Cluster 12
boys
kids
clothing
girls


In [278]:
confidence = cluster_probabilities.max(axis=1)

In [279]:
boundary_docs = np.argsort(confidence)[:10]

In [280]:
for i in boundary_docs:
    print("\nDocument ID:", i)
    print("Cluster probabilities:", cluster_probabilities[i])
    print("\nText:")
    print(clean_documents[i][:400])
    print("\n-----------------------------\n")


Document ID: 4529
Cluster probabilities: [2.8165439e-01 3.8326712e-17 3.8418365e-01 3.1856255e-09 4.0797662e-16
 3.3415529e-01 1.5741982e-09 1.1311786e-12 7.5557193e-07 2.2545211e-40
 6.4216628e-21 4.4274836e-07 5.6624629e-17 8.6525538e-33 3.6423812e-06]

Text:
product prowl by tiger shroff solid men wind cheater category men s wear raincoats and windcheaters raincoats and windcheaters description highlights men windcheater size l color black made of polyester pack of 1 selling price 1 099 product rating 4 0 seller rating 4 8

-----------------------------


Document ID: 1661
Cluster probabilities: [2.8025969e-45 2.3074341e-07 3.8925835e-01 7.0636142e-06 5.2025448e-06
 7.3156203e-10 3.6630401e-01 8.0718723e-32 2.4424390e-01 0.0000000e+00
 4.3407285e-19 5.4644820e-29 4.1325481e-33 1.5692226e-23 1.8252010e-04]

Text:
product recombigen lh test card clear ovum lh one step ovulation test kit pack of 10 test pregnancy test kit 10 tests category sports books and more medical supplies hot wa

In [281]:
semantic_cache = {
    cluster: {"queries": [], "embeddings": [], "results": []}
    for cluster in range(best_k)
}

In [282]:
from sklearn.metrics.pairwise import cosine_similarity

In [283]:
def search_cache(query_embedding, cluster):

    cache = semantic_cache[cluster]

    if len(cache["embeddings"]) == 0:
        return None

    similarities = cosine_similarity(query_embedding, cache["embeddings"])[0]

    best_match = similarities.argmax()
    best_score = similarities[best_match]

    if best_score > SIMILARITY_THRESHOLD:
        return cache["results"][best_match]

    return None

In [284]:
def add_to_cache(query, query_embedding, results):

    semantic_cache["queries"].append(query)

    semantic_cache["embeddings"].append(query_embedding[0])

    semantic_cache["results"].append(results)

In [285]:
def semantic_search(query):

    query_embedding = model.encode([query]).astype("float32")

    cluster = get_query_cluster(query_embedding)

    cached_result = search_cache(query_embedding, cluster)

    if cached_result is not None:
        print("CACHE HIT")
        return cached_result

    print("CACHE MISS")

    distances, indices = index.search(query_embedding, k=5)

    results = [clean_documents[i] for i in indices[0]]

    semantic_cache[cluster]["queries"].append(query)
    semantic_cache[cluster]["embeddings"].append(query_embedding[0])
    semantic_cache[cluster]["results"].append(results)

    return results

In [289]:
def get_query_cluster(query_embedding):

    query_reduced = pca.transform(query_embedding)

    probs = gmm.predict_proba(query_reduced)

    cluster = probs.argmax(axis=1)[0]

    return cluster

In [290]:
SIMILARITY_THRESHOLD = 0.6

In [292]:
result1 = semantic_search("mars exploration missions")

result2 = semantic_search("nasa missions to mars")

CACHE HIT
CACHE HIT


In [293]:
semantic_cache = {
    cluster: {"queries": [], "embeddings": [], "results": []}
    for cluster in range(best_k)
}

In [295]:
def search_cache(query_embedding, cluster):

    cache = semantic_cache.setdefault(
        cluster, {"queries": [], "embeddings": [], "results": []}
    )

    if len(cache["embeddings"]) == 0:
        return None

    similarities = cosine_similarity(query_embedding, cache["embeddings"])[0]

    best_match = similarities.argmax()
    best_score = similarities[best_match]

    if best_score > SIMILARITY_THRESHOLD:
        print(f"CACHE HIT (similarity={best_score:.3f})")
        return cache["results"][best_match]

    return None

In [296]:
def add_to_cache(query, query_embedding, results, cluster):

    semantic_cache[cluster]["queries"].append(query)
    semantic_cache[cluster]["embeddings"].append(query_embedding[0])
    semantic_cache[cluster]["results"].append(results)

In [297]:
def semantic_search(query):

    query_embedding = model.encode([query]).astype("float32")

    cluster = get_query_cluster(query_embedding)

    cached_result = search_cache(query_embedding, cluster)

    if cached_result is not None:
        return cached_result

    print("CACHE MISS")

    distances, indices = index.search(query_embedding, k=5)

    results = [clean_documents[i] for i in indices[0]]

    add_to_cache(query, query_embedding, results, cluster)

    return results

In [299]:
r1 = semantic_search("mars exploration missions")

r2 = semantic_search("nasa missions to mars")

r3 = semantic_search("hockey team players")

print(r3[0][:300])

CACHE HIT (similarity=1.000)
CACHE HIT (similarity=1.000)
CACHE HIT (similarity=1.000)
product foroly fast sling board game for kids and family or interactive foosball board game category bady and kids toys board games description this wooden sling hockey game is the ultimate challenge for adults and kids alike let the fun begin with this unique wooden board that allows everyone to en


In [300]:
pip install fastapi uvicorn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\RACHNA\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [302]:
# imports
import faiss
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer

# load model
model = SentenceTransformer("all-MiniLM-L6-v2")

# load dataset
with open("clean_documents.txt", "r", encoding="utf-8") as f:
    clean_documents = [line.strip() for line in f]

# create embeddings
embeddings = model.encode(clean_documents).astype("float32")

# build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# clustering setup
best_k = 10

semantic_cache = {
    cluster: {"queries": [], "embeddings": [], "results": []}
    for cluster in range(best_k)
}


# ------------------------------------------------
# FASTAPI APP
# ------------------------------------------------

app = FastAPI()


# stats
total_entries = 0
hit_count = 0
miss_count = 0


class QueryRequest(BaseModel):
    query: str


# ------------------------------------------------
# API ROUTES
# ------------------------------------------------


@app.post("/query")
def query_endpoint(request: QueryRequest):

    global hit_count, miss_count, total_entries

    query = request.query

    query_embedding = model.encode([query]).astype("float32")

    cluster = get_query_cluster(query_embedding)

    cached_result = search_cache(query_embedding, cluster)

    if cached_result is not None:
        hit_count += 1

        return {
            "query": query,
            "cache_hit": True,
            "result": cached_result,
            "dominant_cluster": int(cluster),
        }

    miss_count += 1

    distances, indices = index.search(query_embedding, k=5)

    results = [clean_documents[i] for i in indices[0]]

    add_to_cache(query, query_embedding, results, cluster)

    total_entries = sum(len(semantic_cache[c]["queries"]) for c in semantic_cache)

    return {
        "query": query,
        "cache_hit": False,
        "result": results,
        "dominant_cluster": int(cluster),
    }


@app.get("/cache/stats")
def cache_stats():

    total = hit_count + miss_count
    hit_rate = (hit_count / total) * 100 if total > 0 else 0

    return {
        "total_entries": total_entries,
        "hit_count": hit_count,
        "miss_count": miss_count,
        "hit_rate": f"{hit_rate:.2f}%",
    }


@app.delete("/cache")
def clear_cache():

    global semantic_cache, hit_count, miss_count, total_entries

    semantic_cache = {
        cluster: {"queries": [], "embeddings": [], "results": []}
        for cluster in range(best_k)
    }

    hit_count = 0
    miss_count = 0
    total_entries = 0

    return {"message": "Cache cleared"}